<a href="https://colab.research.google.com/github/FANGxPC/LIFELOG_AI/blob/master/LIFELOG_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# # Install specialized versions for Qwen2-VL video support
# !pip install git+https://github.com/huggingface/transformers@21fac7abba2a37fae86106f87fcf9974fd1e3830 accelerate -q
# !pip install qwen-vl-utils[decord] av -q
# !pip install faster-whisper -q


In [2]:
# 1. Install latest transformers from source + 4-bit tools
!pip install -U git+https://github.com/huggingface/transformers bitsandbytes accelerate qwen-vl-utils[decord]
!pip install -U faster-whisper
# 1. KILL THE OLD MODEL (Run this to free up the 13GB)
import torch
import gc
from faster_whisper import WhisperModel
if 'model' in globals():
    del model
    del processor
gc.collect()
torch.cuda.empty_cache()

# 2. LOAD THE 3B MODEL (The "Lite" Powerhouse)
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
# import qwen_vl_utils

# # Increase the internal library limit to match our 10 FPS requirement
# # This stops the WARNING and prevents forced downscaling
# qwen_vl_utils.vision_process.LIMIT = 4194304 # 4.1M pixels (Safe for 8GB VRAM)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("🚀 Loading Qwen2.5-VL-3B-Instruct (Low-VRAM Mode)...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", # ← Changed from 7B to 3B
    quantization_config=quant_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
whisper_model = WhisperModel("tiny.en", device="cuda", compute_type="float16")

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-j2_17iar
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-j2_17iar
  Resolved https://github.com/huggingface/transformers to commit 11b1906d5c0dae39c13270e47cc02c4cde70e548
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 104.3 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11671901 sha256=756e99401d9e5211c6a54082de6a2f9212f30c2fd4ccfb2f25534973c38a14be
  Stored in directory: /tmp/pip-ephem-wheel-cache-6mifsam6/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
from google.colab.output import eval_js
import base64
def capture_10s_moment(filename):
    # We pass the filename in to ensure the main loop controls it
    js_code = f'''
    (async function() {{
      try {{
        const stream = await navigator.mediaDevices.getUserMedia({{video: true, audio: true}});
        const recorder = new MediaRecorder(stream);
        const chunks = [];
        recorder.ondataavailable = (e) => chunks.push(e.data);
        recorder.start();

        console.log("Recording 10s...");
        await new Promise(r => setTimeout(r, 10000));

        recorder.stop();
        return new Promise(r => {{
          recorder.onstop = () => {{
            const blob = new Blob(chunks, {{type: 'video/webm'}});
            const reader = new FileReader();
            reader.readAsDataURL(blob);
            reader.onloadend = () => r(reader.result);
            stream.getTracks().forEach(t => t.stop());
          }}
        }});
      }} catch (err) {{
        return "ERROR:" + err.name;
      }}
    }})()
    '''
    data = eval_js(js_code)

    if data.startswith("ERROR:"):
        return data # Pass the error string back to handle it in Python

    binary = base64.b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return "SUCCESS"

In [8]:
prompt_instruction = """

You are a strict visual auditor.



Return ONLY the 4 sections below.

No introduction.

No conclusion.

No explanations.

No formatting outside the required headings and bullet points.



Follow ALL rules exactly.



--------------------------------------------------

STRICT ANALYSIS RULES

--------------------------------------------------



1. Analyze the ENTIRE video clip from start to end.

3. Use ONLY clearly visible visual evidence.

4. Do NOT assume, infer, or guess beyond observable proof.

5. If evidence is unclear or partially visible, exclude it.




--------------------------------------------------

SECTION 1: Objects (Features + Position)

--------------------------------------------------



• One bullet per clearly visible object.

• Include:

- object name

- color

- material (only if clearly visible)

- readable text / brand (if visible)

- physical condition (if visible)

- exact spatial position (e.g., on table, left side of desk, behind person, on floor, background, near wall, etc.)

- any position change with approximate timestamp (if observed)



If no clear objects:

• No clearly visible objects



--------------------------------------------------

SECTION 2: Movements & Transitions

--------------------------------------------------



• Bullet points only.

• Include:

- human movement

- object movement

- interactions (pick up, place, move, open, close, enter frame, exit frame)

- camera movement (if any)

- approximate timestamps when noticeable



If no movement:

• No visible movement



--------------------------------------------------

SECTION 3: Environment (Maximum 40 words total)

--------------------------------------------------



• Bullet points only.

• Include:

- location type

- lighting condition

- wall appearance

- floor type

- furniture

- background elements

- overall atmosphere



Use concise factual descriptions only.



--------------------------------------------------

SECTION 4: Contextual Event Summary (Maximum 70 words)

--------------------------------------------------



• Bullet points only.

• MUST integrate:

- object positions

- observed movements

- sequence of events over time

• Do NOT speculate about intentions.

• Describe ONLY what can be grounded in evidence.



"""

Checkign thigns


In [12]:
import threading
import queue
import time
import os
from IPython.display import clear_output
import torch
from faster_whisper import WhisperModel

# 1. Setup the Queue
video_queue = queue.Queue()

def ai_processor_worker():
    print("🧠 AI Reality Auditor (3B-Lite) is online...")

    while True:
        video_path = video_queue.get()
        if video_path is None:
            break

        try:
            # 1. Transcription (Base model is lighter than Small)
            segments, _ = whisper_model.transcribe(video_path)

            transcript_lines = []
            for seg in segments:
                transcript_lines.append(
                    f"[{round(seg.start,1)}s - {round(seg.end,1)}s] {seg.text.strip()}"
                  )

            TRANSCRIPT_PLACEHOLDER = "\n".join(transcript_lines)
            # 2. Prepare Messages with HEAVY Pixel Capping
            messages = [{
                "role": "user",
                "content": [
                    {
                        "type": "video",
                        "video": video_path,
                        "fps": 5.0,
                        # This keeps VRAM usage predictable
                        # "min_pixels": 128 * 128,
                        "max_pixels": 448 * 448,
                    },
                    {
    "type": "text",
    "text": f"""
{prompt_instruction}
"""
}
                ]
            }]

            # 3. Process
            text = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt"
            ).to("cuda")

            # 4. Generate (Fast & Greedy)
            with torch.no_grad():
                gen_ids = model.generate(
                    **inputs,
                    max_new_tokens=500,
                    do_sample=False,
                    repetition_penalty=1.1  # Keeps it stable
                )

            output = processor.batch_decode(
                gen_ids[:, inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )[0]

            print(f"\n📡 PULSE: {output[:]}...")

            # 5. AGGRESSIVE CLEANUP
            del inputs, gen_ids, image_inputs, video_inputs
            torch.cuda.empty_cache()

            if os.path.exists(video_path):
                os.remove(video_path)

        except Exception as e:
            print(f"❌ Error: {e}")
            torch.cuda.empty_cache()

        finally:
            video_queue.task_done()

# 2. Start the Background Processor
worker_thread = threading.Thread(
    target=ai_processor_worker,
    daemon=True
)
worker_thread.start()

# 3. Main Loop (The Continuous Recorder)
def start_infinite_parallel_lifelog():
    print("🚀 INFINITE SYNC RESTARTED")
    loop = 0

    try:
        while True:
            loop += 1
            filename = f"vid_{int(time.time())}.webm"

            # Attempt to record
            status = capture_10s_moment(filename)

            if status == "SUCCESS":
                video_queue.put(filename)

                # Keep only 20 files at a time just in case cleanup fails
                if loop % 10 == 0:
                    torch.cuda.empty_cache()
            else:
                print(f"⚠️ Camera Glitch ({status}). Retrying in 2s...")
                time.sleep(2)

    except KeyboardInterrupt:
        print("\n🛑 Loop manually stopped.")
        video_queue.put(None)

# Run this to start again
start_infinite_parallel_lifelog()

🧠 AI Reality Auditor (3B-Lite) is online...
🚀 INFINITE SYNC RESTARTED

📡 PULSE: **SECTION 1: Objects**

- **Person**: Wearing a blue shirt, seated against a plain white wall.
- **Wall**: Plain white wall with a light switch on the right side.
- **Bedspread**: Pink and white patterned bedspread.
- **Clothes**: Black clothes hanging on the bed.
- **Mirror**: Wooden-framed mirror on the right side of the wall.
- **Light Switch**: White light switch on the wall.
- **Cable**: Thin cable running along the wall.

**SECTION 2: Movements & Transitions**

- **Human Movement**: Person remains seated throughout the video.
- **Object Movement**: None significant changes in object positions noted.

**SECTION 3: Environment**

- **Location Type**: Bedroom setting.
- **Lighting Condition**: Natural light coming from an unseen source.
- **Wall Appearance**: Plain white wall.
- **Floor Type**: Not explicitly visible but likely carpeted based on context.
- **Furniture**: Bed with pink and white spread.
-